# Imports

Installs

In [51]:
# ! pip install -U ipython-sql prettytable==3.9.0

Load extension

In [52]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


Load data

In [53]:
%sql sqlite:///data.sqlite

Get table names 

In [54]:
%%sql
SELECT name 
FROM sqlite_master 
WHERE type='table';

   sqlite:///data.slite
 * sqlite:///data.sqlite
Done.


name
orderdetails
payments
offices
customers
orders
productlines
products
employees


# Grouping Data with SQL

## Introduction to GROUP BY and Aggregate Functions

Aggregate functions in SQL are used to summarize data. They help calculate values like totals, averages, minimums, and maximums.

Common aggregate functions:

- `COUNT()` – Counts rows  
- `MIN()` – Returns the smallest value  
- `MAX()` – Returns the largest value  
- `SUM()` – Returns the total  
- `AVG()` – Returns the average  

The `GROUP BY` clause groups rows into summary rows and returns one result per group. It is typically used together with aggregate functions.

Example use case:
- Counting customers per country
- Comparing total sales across regions

In [55]:
%%sql
SELECT COUNT(*) AS employeeCount FROM employees

   sqlite:///data.slite
 * sqlite:///data.sqlite
Done.


employeeCount
23


In [75]:
%%sql
select * from products limit 2

   sqlite:///data.slite
 * sqlite:///data.sqlite
Done.


productCode,productName,productLine,productScale,productVendor,productDescription,quantityInStock,buyPrice,MSRP
S10_1678,1969 Harley Davidson Ultimate Chopper,Motorcycles,1:10,Min Lin Diecast,"This replica features working kickstand, front suspension, gear-shift lever, footbrake lever, drive chain, wheels and steering. All parts are particularly delicate due to their precise scale and require special care and attention.",7933,48.81,95.70
S10_1949,1952 Alpine Renault 1300,Classic Cars,1:10,Classic Metal Creations,Turnable front wheels; steering function; detailed interior; detailed engine; opening hood; opening trunk; opening doors; and detailed chassis.,7305,98.58,214.30


In [57]:
%%sql
SELECT SUM(quantityInStock) AS stockCount FROM products 

   sqlite:///data.slite
 * sqlite:///data.sqlite
Done.


stockCount
555131


## What is GROUP BY?

`GROUP BY` is used in SQL to **combine rows that have the same values in one or more columns** into summary rows.

It is usually used together with **aggregate functions**

## Basic Syntax

```sql
SELECT column_name, AGGREGATE_FUNCTION(column_name)
FROM table_name
GROUP BY column_name;

In [58]:
%%sql
SELECT country,Count(country)
FROM customers
GROUP BY country LIMIT 5;

   sqlite:///data.slite
 * sqlite:///data.sqlite
Done.


country,Count(country)
Australia,5
Austria,2
Belgium,2
Canada,3
Denmark,2



## Alternative GROUP BY Syntax

Instead of grouping by a column name, you can group by the column position in the `SELECT` statement.

Example:

```sql
SELECT country, COUNT(*)
FROM customers
GROUP BY 1;
```

`GROUP BY 1` refers to the first selected column (`country`).



In [59]:
%%sql
SELECT country,Count(country)
FROM customers
GROUP BY 1 LIMIT 5;

   sqlite:///data.slite
 * sqlite:///data.sqlite
Done.


country,Count(country)
Australia,5
Austria,2
Belgium,2
Canada,3
Denmark,2


## Aliasing

An alias is a temporary name given to a column or table.

Example:

```sql
SELECT country, COUNT(*) AS customer_count
FROM customers
GROUP BY country;
```

Important notes:

- Aliases only exist during the query.
- The keyword `AS` is optional in SQLite.
- Aliases improve readability, especially when using aggregates.


In [60]:
%%sql
SELECT country, COUNT(*) AS customer_count
FROM customers
GROUP BY country LIMIT 5;


   sqlite:///data.slite
 * sqlite:///data.sqlite
Done.


country,customer_count
Australia,5
Austria,2
Belgium,2
Canada,3
Denmark,2


## Multiple Aggregations Example

We calculated payment statistics per customer:

```sql
SELECT
    customerNumber,
    COUNT(*) AS number_payments,
    MIN(amount) AS min_purchase,
    MAX(amount) AS max_purchase,
    AVG(amount) AS avg_purchase,
    SUM(amount) AS total_spent
FROM payments
GROUP BY customerNumber;
```

This returned:

- Number of payments  
- Minimum purchase  
- Maximum purchase  
- Average purchase  
- Total spent  

All grouped by `customerNumber`.


In [61]:
%%sql
SELECT
    customerNumber,
    COUNT(*) AS number_payments,
    MIN(amount) AS min_purchase,
    MAX(amount) AS max_purchase,
    AVG(amount) AS avg_purchase,
    SUM(amount) AS total_spent
FROM payments
GROUP BY customerNumber LIMIT 5;

   sqlite:///data.slite
 * sqlite:///data.sqlite
Done.


customerNumber,number_payments,min_purchase,max_purchase,avg_purchase,total_spent
103,3,14571.44,6066.78,7438.12,22314.36
112,3,14191.12,33347.88,26726.993333333332,80180.98
114,4,44894.74,82261.22,45146.2675,180585.07
119,3,19501.82,49523.67,38983.22666666666,116949.68
121,4,1491.38,50218.95,26056.1975,104224.79



## Filtering with WHERE (Before Aggregation)

The `WHERE` clause filters rows before grouping happens.

Example: Filtering payments made in 2004.

```sql
SELECT
    customerNumber,
    COUNT(*) AS number_payments,
    MIN(amount) AS min_purchase,
    MAX(amount) AS max_purchase,
    AVG(amount) AS avg_purchase,
    SUM(amount) AS total_spent
FROM payments
WHERE strftime('%Y', paymentDate) = '2004'
GROUP BY customerNumber;
```

Key points:

- Filtering happens before aggregation.
- Results change because only filtered rows are included.
- Columns used in `WHERE` do not need to appear in `SELECT`.


In [62]:
%%sql
SELECT
    customerNumber,
    COUNT(*) AS number_payments,
    MIN(amount) AS min_purchase,
    MAX(amount) AS max_purchase,
    AVG(amount) AS avg_purchase,
    SUM(amount) AS total_spent
FROM payments
WHERE strftime('%Y', paymentDate) = '2004'
GROUP BY customerNumber LIMIT 5;

   sqlite:///data.slite
 * sqlite:///data.sqlite
Done.


customerNumber,number_payments,min_purchase,max_purchase,avg_purchase,total_spent
103,2,1676.14,6066.78,3871.46,7742.92
112,2,14191.12,33347.88,23769.5,47539.0
114,2,44894.74,82261.22,63577.979999999996,127155.95999999999
119,2,19501.82,47924.19,33713.005000000005,67426.01000000001
121,2,17876.32,34638.14,26257.23,52514.46


## The HAVING Clause (After Aggregation)

The `HAVING` clause filters results after grouping and aggregation.

Example:

```sql
SELECT
    customerNumber,
    COUNT(*) AS number_payments,
    MIN(amount) AS min_purchase,
    MAX(amount) AS max_purchase,
    AVG(amount) AS avg_purchase,
    SUM(amount) AS total_spent
FROM payments
GROUP BY customerNumber
HAVING AVG(amount) > 50000;
```

Key difference:

- `WHERE` → Filters rows before grouping  
- `HAVING` → Filters aggregated results after grouping  

In most SQL dialects, you must repeat the aggregate function in `HAVING` instead of using its alias.


In [63]:
%%sql
SELECT
    customerNumber,
    COUNT(*) AS number_payments,
    MIN(amount) AS min_purchase,
    MAX(amount) AS max_purchase,
    AVG(amount) AS avg_purchase,
    SUM(amount) AS total_spent
FROM payments
GROUP BY customerNumber
HAVING AVG(amount) > 50000 LIMIT 5;

   sqlite:///data.slite
 * sqlite:///data.sqlite
Done.


customerNumber,number_payments,min_purchase,max_purchase,avg_purchase,total_spent
124,9,101244.59,85410.87,64909.804444444446,584188.24
141,13,116208.40,65071.26,55056.844615384616,715738.98
239,1,80375.24,80375.24,80375.24,80375.24
298,2,47375.92,61402.00,54388.96,108777.92
321,2,46781.66,85559.12,66170.39,132340.78


## Combining WHERE and HAVING

We can combine both clauses for more complex logic.

Example: Customers who made at least 2 purchases over 50,000.

```sql
SELECT
    customerNumber,
    COUNT(*) AS number_payments,
    MIN(amount) AS min_purchase,
    MAX(amount) AS max_purchase,
    AVG(amount) AS avg_purchase,
    SUM(amount) AS total_spent
FROM payments
WHERE amount > 50000
GROUP BY customerNumber
HAVING COUNT(*) >= 2;
```

Process:

1. `WHERE` filters payments over 50,000  
2. `GROUP BY` aggregates by customer  
3. `HAVING` filters customers with at least 2 such payments  


In [64]:
%%sql
SELECT
    customerNumber,
    COUNT(*) AS number_payments,
    MIN(amount) AS min_purchase,
    MAX(amount) AS max_purchase,
    AVG(amount) AS avg_purchase,
    SUM(amount) AS total_spent
FROM payments
WHERE amount > 50000
GROUP BY customerNumber
HAVING COUNT(*) >= 2 LIMIT 5;

   sqlite:///data.slite
 * sqlite:///data.sqlite
Done.


customerNumber,number_payments,min_purchase,max_purchase,avg_purchase,total_spent
103,3,14571.44,6066.78,7438.12,22314.36
112,3,14191.12,33347.88,26726.993333333332,80180.98
114,4,44894.74,82261.22,45146.2675,180585.07
119,3,19501.82,49523.67,38983.22666666666,116949.68
121,4,1491.38,50218.95,26056.1975,104224.79


## Using ORDER BY and LIMIT

We can also sort and limit aggregated results.

Example:

```sql
SELECT
    customerNumber,
    COUNT(*) AS number_payments,
    MIN(amount) AS min_purchase,
    MAX(amount) AS max_purchase,
    AVG(amount) AS avg_purchase,
    SUM(amount) AS total_spent
FROM payments
WHERE amount > 50000
GROUP BY customerNumber
HAVING COUNT(*) >= 2
ORDER BY total_spent
LIMIT 1;
```

This finds the customer with the lowest total spending among those meeting the criteria.


In [65]:
%%sql
SELECT
    customerNumber,
    COUNT(*) AS number_payments,
    MIN(amount) AS min_purchase,
    MAX(amount) AS max_purchase,
    AVG(amount) AS avg_purchase,
    SUM(amount) AS total_spent
FROM payments
WHERE amount > 50000
GROUP BY customerNumber
HAVING COUNT(*) >= 2
ORDER BY total_spent
LIMIT 1;

   sqlite:///data.slite
 * sqlite:///data.sqlite
Done.


customerNumber,number_payments,min_purchase,max_purchase,avg_purchase,total_spent
219,2,3452.75,4465.85,3959.3,7918.6


# ORDER BY

By default, SQL returns rows in the order they are stored in the database. This order may not be meaningful.

To sort results, we use:

```sql
SELECT column(s)
FROM table_name
ORDER BY column_name;
```

Example:

```sql
SELECT *
FROM products
ORDER BY productName;
```

In [68]:
%%sql
SELECT *
FROM products
LIMIT 2;

   sqlite:///data.slite
 * sqlite:///data.sqlite
Done.


productCode,productName,productLine,productScale,productVendor,productDescription,quantityInStock,buyPrice,MSRP
S10_1678,1969 Harley Davidson Ultimate Chopper,Motorcycles,1:10,Min Lin Diecast,"This replica features working kickstand, front suspension, gear-shift lever, footbrake lever, drive chain, wheels and steering. All parts are particularly delicate due to their precise scale and require special care and attention.",7933,48.81,95.70
S10_1949,1952 Alpine Renault 1300,Classic Cars,1:10,Classic Metal Creations,Turnable front wheels; steering function; detailed interior; detailed engine; opening hood; opening trunk; opening doors; and detailed chassis.,7305,98.58,214.30


In [69]:
%%sql
SELECT *
FROM products
ORDER BY productName LIMIT 2;

   sqlite:///data.slite
 * sqlite:///data.sqlite
Done.


productCode,productName,productLine,productScale,productVendor,productDescription,quantityInStock,buyPrice,MSRP
S18_3136,18th Century Vintage Horse Carriage,Vintage Cars,1:18,Red Start Diecast,"Hand crafted diecast-like metal horse carriage is re-created in about 1:18 scale of antique horse carriage. This antique style metal Stagecoach is all hand-assembled with many different parts.This collectible metal horse carriage is painted in classic Red, and features turning steering wheel and is entirely hand-finished.",5992,60.74,104.72
S24_2011,18th century schooner,Ships,1:24,Carousel DieCast Legends,"All wood with canvas sails. Many extras including rigging, long boats, pilot house, anchors, etc. Comes with 4 masts, all square-rigged.",1898,82.34,122.89


## Ascending and Descending Order

- `ASC` → Ascending order (default)
- `DESC` → Descending order

Example:

```sql
ORDER BY productName ASC;
ORDER BY productName DESC;
```


In [ ]:
%%sql
SELECT *
FROM products
ORDER BY productName ASC LIMIT 2;

In [ ]:
%%sql
SELECT *
FROM products
ORDER BY productName DESC LIMIT 2;


# Sorting by Multiple Columns

You can sort by more than one column.

Example:

```sql
SELECT productVendor, productName, MSRP
FROM products
ORDER BY productVendor, productName;
```

How it works:
1. Sort by `productVendor`
2. If vendors are the same, use `productName` as a tiebreaker

Important rule:
- Columns with fewer unique values should come first.
- Columns with more unique values should come later.

In [71]:
%%sql
SELECT productVendor, productName, MSRP
FROM products
ORDER BY productVendor, productName LIMIT 5;

   sqlite:///data.slite
 * sqlite:///data.sqlite
Done.


productVendor,productName,MSRP
Autoart Studio Design,1900s Vintage Bi-Plane,68.51
Autoart Studio Design,1932 Model A Ford J-Coupe,127.13
Autoart Studio Design,1937 Horch 930V Limousine,65.75
Autoart Studio Design,1962 Volkswagen Microbus,127.79
Autoart Studio Design,1968 Ford Mustang,194.57


We can check uniqueness using:

```sql
SELECT COUNT(DISTINCT productVendor),
       COUNT(DISTINCT productName)
FROM products;
```

In [72]:
%%sql
SELECT COUNT(DISTINCT productVendor),
       COUNT(DISTINCT productName)
FROM products

   sqlite:///data.slite
 * sqlite:///data.sqlite
Done.


COUNT(DISTINCT productVendor),COUNT(DISTINCT productName)
13,110



# Type Casting in ORDER BY

Sometimes data appears numeric but is stored as text.

Example problem:

```sql
ORDER BY quantityInStock;
```

If `quantityInStock` is stored as text, it will sort alphabetically instead of numerically.

Solution: Use `CAST`

```sql
ORDER BY CAST(quantityInStock AS INTEGER);
```

This forces proper numeric sorting.


In [74]:
%%sql
SELECT *
FROM products
ORDER BY quantityInStock LIMIT 3

   sqlite:///data.slite
 * sqlite:///data.sqlite
Done.


productCode,productName,productLine,productScale,productVendor,productDescription,quantityInStock,buyPrice,MSRP
S24_1046,1970 Chevy Chevelle SS 454,Classic Cars,1:24,Unimax Art Galleries,"This model features rotating wheels, working streering system and opening doors. All parts are particularly delicate due to their precise scale and require special care and attention. It should not be picked up by the doors, roof, hood or trunk.",1005,49.24,73.49
S50_1392,Diamond T620 Semi-Skirted Tanker,Trucks and Buses,1:50,Highway 66 Mini Classics,This limited edition model is licensed and perfectly scaled for Lionel Trains. The Diamond T620 has been produced in solid precision diecast and painted with a fire baked enamel finish. It comes with a removable tanker and is a perfect model to add authenticity to your static train or car layout or to just have on display.,1016,68.29,115.75
S12_3891,1969 Ford Falcon,Classic Cars,1:12,Second Gear Diecast,Turnable front wheels; steering function; detailed interior; detailed engine; opening hood; opening trunk; opening doors; and detailed chassis.,1049,83.05,173.02


In [73]:
%%sql
SELECT *
FROM products
ORDER BY CAST(quantityInStock AS INTEGER) LIMIT 3

   sqlite:///data.slite
 * sqlite:///data.sqlite
Done.


productCode,productName,productLine,productScale,productVendor,productDescription,quantityInStock,buyPrice,MSRP
S24_2000,1960 BSA Gold Star DBD34,Motorcycles,1:24,Highway 66 Mini Classics,Detailed scale replica with working suspension and constructed from over 70 parts,15,37.32,76.17
S12_1099,1968 Ford Mustang,Classic Cars,1:12,Autoart Studio Design,"Hood, doors and trunk all open to reveal highly detailed interior features. Steering wheel actually turns the front wheels. Color dark green.",68,95.34,194.57
S32_4289,1928 Ford Phaeton Deluxe,Vintage Cars,1:32,Highway 66 Mini Classics,"This model features grille-mounted chrome horn, lift-up louvered hood, fold-down rumble seat, working steering system",136,33.02,68.79
